In [3]:
import pandas as pd

# Load ranked baseline(EGFR)
ranked_baseline = pd.read_parquet("../../processed/ENSG00000146648_composite_baseline.parquet")

# Load the cleaned genomic signatures
signatures = pd.read_parquet("../../processed/signatures_clean.parquet")

print("Ranked Baseline Shape:", ranked_baseline.shape)
print("Signatures Shape:", signatures.shape)
display(signatures.head())

Ranked Baseline Shape: (1581, 10)
Signatures Shape: (1955, 6)


,MSIScore,LoHFraction,WGD,CIN,Ploidy,Aneuploidy
ModelID,,,,,,
ACH-000839,3.68,0.107443,1.0,0.502634,3.158291,20.0
ACH-000041,2.21,0.130089,1.0,0.523865,3.236089,19.0
ACH-002046,2.87,0.222342,1.0,0.679772,3.326715,30.0
ACH-002048,2.48,NaN,NaN,NaN,NaN,NaN
ACH-000042,2.07,NaN,NaN,NaN,NaN,NaN


In [4]:
def apply_genomic_exclusion(ranked_df, sig_df, msi_threshold=3.0, cin_threshold=0.5):
    """
    Filters out cell lines with high genomic instability.
    """
    # Merge the genomic signatures onto our ranked list
    merged = pd.merge(ranked_df, sig_df, left_on="ACH_ID", right_on="ModelID", how="left")

    # Identify lines that breach our exclusion thresholds
    exclude_cond = (merged["MSIScore"] > msi_threshold) | (merged["CIN"] > cin_threshold)
    exclude_cond = exclude_cond.fillna(False) # Avoid accidentally drop cell lines just because they are missing signature data

    # Keep only the cell lines that DO NOT meet the exclusion criteria
    filtered_df = merged[~exclude_cond].copy()
    # Reset index for a clean table
    filtered_df = filtered_df.reset_index(drop=True)
    
    print(f"Original ranked candidates: {len(ranked_df)}")
    print(f"Cell lines excluded (MSI > {msi_threshold} or CIN > {cin_threshold}): {exclude_cond.sum()}")
    print(f"Remaining viable candidates: {len(filtered_df)}")
    
    return filtered_df

# Run the filter on our EGFR baseline
viable_candidates = apply_genomic_exclusion(ranked_baseline, signatures)

# Display the refined top 10
cols_to_show = ["ACH_ID", "cell_line_name", "composite_percentile", "MSIScore", "CIN", "confidence_score"]
display(viable_candidates[cols_to_show].head(10))

Original ranked candidates: 1581
Cell lines excluded (MSI > 3.0 or CIN > 0.5): 978
Remaining viable candidates: 603


,ACH_ID,cell_line_name,composite_percentile,MSIScore,CIN,confidence_score
0,ACH-002680,170MGBA,99.459094,2.18,0.478358,0.333333
1,ACH-001649,SHMAC5,99.391481,2.35,NaN,0.333333
2,ACH-000170,PRECLH,99.141832,NaN,NaN,0.625275
3,ACH-000916,NCIH1573,99.075852,0.81,0.408237,0.390583
4,ACH-000317,TUHR14TKB,98.712748,2.99,NaN,0.606448
5,ACH-000029,HCC827GR5,98.407448,2.61,NaN,0.464435
6,ACH-001442,A388,98.377282,1.91,0.225603,0.333333
7,ACH-001090,HN,97.828054,NaN,NaN,0.333333
8,ACH-000715,SNU1214,97.391689,2.21,0.300039,0.579315
9,ACH-000642,HMEL,97.380767,NaN,NaN,0.594905


In [19]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def recommend_similar_cell_lines(target_ach, viable_df, sig_df, top_n=10):
    """
    Finds the most biologically similar cell lines from the viable list compared to a specific target cell line.
    """
    feature_cols = ["MSIScore", "CIN"]
    
    target_profile = sig_df[sig_df.index == target_ach][feature_cols]
    
    if target_profile.empty:
        return f"Cannot compute similarity: Target {target_ach} not found."
    
    # Fill missing signature data with 0 instead of crashing
    target_vector = target_profile.fillna(0)
    
    # Get the profiles of all viable cell lines
    viable_features = viable_df.copy()
    
    # Prevent comparing the cell line to itself
    viable_features = viable_features[viable_features["ACH_ID"] != target_ach].copy()
    
    if viable_features.empty:
         return "No viable cell lines with sufficient data to compare."

    # Fill NaNs in the viable candidates with 0 as well so the math works
    viable_vectors = viable_features[feature_cols].fillna(0)

    # Calculate Cosine Similarity
    similarities = cosine_similarity(target_vector, viable_vectors)
    
    # Add the similarity scores to viable dataframe
    viable_features["similarity_to_target"] = similarities[0]
    
    # Sort to find the highest similarity scores
    top_similar = viable_features.sort_values(by="similarity_to_target", ascending=False).head(top_n)
    
    return top_similar[["ACH_ID", "cell_line_name", "composite_percentile", "similarity_to_target"]]

# Test with HCC827
target_cell_line = "ACH-000029" 
similar_recommendations = recommend_similar_cell_lines(target_cell_line, viable_candidates, signatures)

print(f"Top alternative recommendations for {target_cell_line}:")
display(similar_recommendations)

Top alternative recommendations for ACH-000029:


,ACH_ID,cell_line_name,composite_percentile,similarity_to_target
153,ACH-000785,NCIH2126,69.003940,1.0
359,ACH-002693,S462,34.651792,1.0
116,ACH-000075,U87MG,73.622827,1.0
236,ACH-000617,OVCAR4,56.489549,1.0
360,ACH-000431,NCIH1694,34.423987,1.0
231,ACH-000017,SKBR3,57.422871,1.0
229,ACH-000223,HCC1937,57.760056,1.0
123,ACH-000776,ONS76,72.558641,1.0
124,ACH-000719,RMGI,72.483747,1.0
228,ACH-002465,RPE1SS119,57.809331,1.0
